# Pipe pre-check

In [1]:
import pandas as pd
from src.pipeline.rul_pipeline import RULPipeline

df = pd.read_csv('data/clean/data_motor_1.csv')
df.insert(0, 'unit_number', 1)

pipeline = RULPipeline(window_size=30, clipping_threshold=125, verbose=True)
motor_windows = pipeline.transform(df)

print(f"\nMotor 1:")
print(f"  X_windows shape: {motor_windows[1]['X_windows'].shape}")
print(f"  Eventos: {motor_windows[1]['evento'].sum()}")

[Nodo 2] 1 motors, 163 windows — 0.01s
[Nodo 3] 192 features per window — 2.88s
[Total]  2.89s

Motor 1:
  X_windows shape: (163, 192)
  Eventos: 1


# BN pre check

# SVR pre-check

In [1]:
"""Pipeline validation — Orchestrator (RULPipeline).

Validates the RULPipeline orchestrator on a real GGS fold:
- Timing per stage
- Output shapes and alignment
- y_rul reconstruction
- Ready for GGS consumption

Run from project root or paste into Jupyter notebook.
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from src.dataset_manager import DatasetManager
from src.pipeline.rul_pipeline import RULPipeline

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WINDOW_SIZE   = 30
CLIPPING      = 125
N_COMPONENTS  = 10
N_FOLDS       = 5
FOLD_IDX      = 0

print("=" * 60)
print("PIPELINE VALIDATION — RULPipeline Orchestrator")
print("=" * 60)

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
t_total = time.perf_counter()
m_train, _ = DatasetManager.split_dataset()

dfs = []
for idx in m_train:
    df_motor = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
    df_motor.insert(0, 'unit_number', idx)
    dfs.append(df_motor)
df_all = pd.concat(dfs, ignore_index=True)

# GGS separation
X_df_full = df_all.drop(columns=['RUL'])
y_df_full = df_all[['unit_number', 'time_in_cycles', 'RUL']]
groups_full = df_all['unit_number'].to_numpy()

# ---------------------------------------------------------------------------
# Fold split
# ---------------------------------------------------------------------------
gkf = GroupKFold(n_splits=N_FOLDS)
splits = list(gkf.split(X_df_full, groups=groups_full))
train_idx, val_idx = splits[FOLD_IDX]

X_train_df = X_df_full.iloc[train_idx].reset_index(drop=True)
X_val_df   = X_df_full.iloc[val_idx].reset_index(drop=True)
y_train_df = y_df_full.iloc[train_idx].reset_index(drop=True)
y_val_df   = y_df_full.iloc[val_idx].reset_index(drop=True)

print(f"\nFold {FOLD_IDX}:")
print(f"  Train: {X_train_df['unit_number'].nunique()} motors, {len(X_train_df)} rows")
print(f"  Val:   {X_val_df['unit_number'].nunique()} motors, {len(X_val_df)} rows")

# ---------------------------------------------------------------------------
# RULPipeline — fit_transform on training
# ---------------------------------------------------------------------------
print(f"\n{'─'*60}")
print("fit_transform (training fold)")
print('─'*60)

pipeline = RULPipeline(
    window_size=WINDOW_SIZE,
    clipping_threshold=CLIPPING,
    n_components=N_COMPONENTS,
    verbose=True,
)

t0 = time.perf_counter()
X_train, y_rul_tr, t_stop_tr, evento_tr, groups_tr = pipeline.fit_transform(
    X_train_df, y_train_df
)
t_fit = time.perf_counter() - t0

print(f"\n  Total fit_transform: {t_fit:.1f}s")
print(f"\n  Output validation:")
print(f"    X_train shape:      {X_train.shape}")
print(f"    y_rul_tr NaN:       {np.isnan(y_rul_tr).any()}")
print(f"    y_rul_tr range:     {y_rul_tr.min():.1f} → {y_rul_tr.max():.1f}")
print(f"    t_stop_tr range:    {t_stop_tr.min():.0f} → {t_stop_tr.max():.0f}")
print(f"    evento_tr sum:      {evento_tr.sum()} (train motors with fallo)")
print(f"    groups_tr unique:   {len(np.unique(groups_tr))} motors")
print(f"    X_train NaN:        {np.isnan(X_train).any()}")
print(f"    X_train range:      {X_train.min():.3f} → {X_train.max():.3f}")

evr = pipeline.explained_variance_ratio()
print(f"\n  PCA explained variance:")
print(f"    Per component: {evr.round(3)}")
print(f"    Cumulative:    {evr.cumsum()[-1]:.3f}")

# ---------------------------------------------------------------------------
# RULPipeline — transform on validation
# ---------------------------------------------------------------------------
print(f"\n{'─'*60}")
print("transform (validation fold)")
print('─'*60)

t0 = time.perf_counter()
X_val, y_rul_val, t_stop_val, evento_val, groups_val = pipeline.transform(
    X_val_df, y_val_df
)
t_transform = time.perf_counter() - t0

print(f"\n  Total transform: {t_transform:.1f}s")
print(f"\n  Output validation:")
print(f"    X_val shape:        {X_val.shape}")
print(f"    y_rul_val NaN:      {np.isnan(y_rul_val).any()}")
print(f"    y_rul_val range:    {y_rul_val.min():.1f} → {y_rul_val.max():.1f}")
print(f"    evento_val sum:     {evento_val.sum()} (val motors with fallo)")
print(f"    groups_val unique:  {len(np.unique(groups_val))} motors")
print(f"    X_val NaN:          {np.isnan(X_val).any()}")

# ---------------------------------------------------------------------------
# GGS readiness check
# ---------------------------------------------------------------------------
print(f"\n{'─'*60}")
print("GGS readiness check")
print('─'*60)
print(f"  ✅ X_train ready:    {X_train.shape} — model.fit(X_train, y_rul_tr)")
print(f"  ✅ X_val ready:      {X_val.shape}   — model.predict(X_val)")
print(f"  ✅ y_rul available:  train={not np.isnan(y_rul_tr).any()}, val={not np.isnan(y_rul_val).any()}")
print(f"  ✅ groups ready:     {groups_tr.shape} — GroupKFold compatible")
print(f"  ✅ pipeline fitted:  {pipeline.is_fitted_}")

print(f"\n{'='*60}")
print(f"Total time: {time.perf_counter()-t_total:.1f}s")
print("Pipeline is ready for GGS.")

PIPELINE VALIDATION — RULPipeline Orchestrator
Training engines: 140
Test engines: 60

Fold 0:
  Train: 112 motors, 19062 rows
  Val:   28 motors, 4762 rows

────────────────────────────────────────────────────────────
fit_transform (training fold)
────────────────────────────────────────────────────────────
  Nodo 2 (windowing): 4.73s
  Nodo 3 (features): 330.31s
  Nodo 4 (PCA): 0.97s

  Total fit_transform: 336.2s

  Output validation:
    X_train shape:      (15814, 10)
    y_rul_tr NaN:       False
    y_rul_tr range:     0.0 → 125.0
    t_stop_tr range:    30 → 287
    evento_tr sum:      57 (train motors with fallo)
    groups_tr unique:   112 motors
    X_train NaN:        False
    X_train range:      -12.562 → 27.321

  PCA explained variance:
    Per component: [0.304 0.136 0.047 0.015 0.013 0.012 0.012 0.012 0.011 0.011]
    Cumulative:    0.572

────────────────────────────────────────────────────────────
transform (validation fold)
─────────────────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
from src.dataset_manager import DatasetManager
from src.ggs_training_manager import GGSTrainingManager
from src.models.negative_binomial import NegativeBinomialPiecewise
from src.models.svr_model import SVRModel

# ---------------------------------------------------------------------------
# Preparar datos
# ---------------------------------------------------------------------------
m_train, _ = DatasetManager.split_dataset()

dfs = []
for idx in m_train:
    df = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
    df.insert(0, 'unit_number', idx)
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

X_df = df_all.drop(columns=['RUL'])
y_df = df_all[['unit_number', 'time_in_cycles', 'RUL']]
groups = df_all['unit_number'].to_numpy()

# ---------------------------------------------------------------------------
# GGS — Negative Binomial
# ---------------------------------------------------------------------------
manager_nb = GGSTrainingManager(
    model_class=NegativeBinomialPiecewise,
    X_df=X_df, y_df=y_df, groups=groups,
)
manager_nb.group_grid_search(
    param_grid={
        'window_size':        [20, 30],
        'n_components':       [5, 10],
        'clipping_threshold': [125],
        'alpha':              [0.5, 1.0],
        'link_type':          ['log'],
    },
    n_folds=3,
)
display(manager_nb.get_ggs_results(top_n=5))

## Feature Exploration

In [2]:
"""Feature exploration script — comparing feature sets for RUL pipeline.

Evaluates four feature set configurations on motor 1:
    A — Current:     median, abs_energy, q25, q75 + trend features
    B — RMS swap:    replaces abs_energy with rms
    C — +FFT:        B + fft coefficients (0-5)
    D — +Entropy:    B + permutation entropy

For each configuration measures:
    1. Extraction time (Nodo 3 cost)
    2. Feature range and std (scale before PCA)
    3. PCA explained variance with n_components=10 and 15
    4. Correlation between rms and abs_energy (redundancy check)

Run from project root or paste into Jupyter notebook.
"""

import time
import warnings
import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy

from src.pipeline.windowing import build_windows, flatten_windows
from src.pipeline.feature_extraction import (
    _compute_features,
    STATISTICAL_FEATURES,
    TREND_FEATURES,
    ALL_FEATURES,
    MotorWindows,
)
from src.pipeline.dim_reduction import DimReducer

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WINDOW_SIZE  = 30
CLIPPING     = 125
MOTOR_ID     = 1

print("=" * 60)
print("FEATURE EXPLORATION — Motor 1")
print("=" * 60)

# ---------------------------------------------------------------------------
# Load data and build windows
# ---------------------------------------------------------------------------
df = pd.read_csv(f'data/clean/data_motor_{MOTOR_ID}.csv')
df.insert(0, 'unit_number', MOTOR_ID)
X_df = df.drop(columns=['RUL'])

motor_windows = build_windows(X_df, WINDOW_SIZE, CLIPPING)
X_3d = motor_windows[MOTOR_ID]['X_windows']  # (n_windows, window_size, n_sensors)
n_windows, window_size, n_sensors = X_3d.shape
print(f"\nWindows: {n_windows}, window_size: {window_size}, sensors: {n_sensors}")

# ---------------------------------------------------------------------------
# Helper — compute rms
# ---------------------------------------------------------------------------
def compute_rms(X: np.ndarray) -> np.ndarray:
    """RMS = sqrt(mean(x²)) over window axis. Shape: (n_windows, n_sensors)."""
    return np.sqrt(np.mean(X ** 2, axis=1))

def compute_fft_coefs(X: np.ndarray, n_coefs: int = 5) -> np.ndarray:
    """FFT absolute coefficients 0..n_coefs-1 over window axis.
    Shape: (n_windows, n_coefs * n_sensors)."""
    blocks = []
    for k in range(n_coefs):
        fft_vals = np.fft.rfft(X, axis=1)
        coef_k = np.abs(fft_vals[:, k, :])  # (n_windows, n_sensors)
        blocks.append(coef_k)
    return np.concatenate(blocks, axis=1)

def compute_permutation_entropy(X: np.ndarray, tau: int = 1, dim: int = 3) -> np.ndarray:
    """Permutation entropy per window per sensor.
    Shape: (n_windows, n_sensors). Slow — Python loop."""
    result = np.zeros((n_windows, n_sensors))
    for w in range(n_windows):
        for s in range(n_sensors):
            series = X[w, :, s]
            # Build ordinal patterns
            n = len(series) - (dim - 1) * tau
            if n <= 0:
                continue
            patterns = []
            for i in range(n):
                sub = series[i:i + dim * tau:tau]
                patterns.append(tuple(np.argsort(sub)))
            # Count pattern frequencies
            from collections import Counter
            counts = Counter(patterns)
            total = sum(counts.values())
            probs = np.array([v / total for v in counts.values()])
            result[w, s] = scipy_entropy(probs, base=2)
    return result

# ---------------------------------------------------------------------------
# Feature set definitions
# ---------------------------------------------------------------------------
BASE_TREND = TREND_FEATURES  # slope, rvalue, autocorr x3, partial_autocorr x3

FEATURE_SETS = {
    'A_current':  ('median', 'abs_energy', 'q25', 'q75') + tuple(BASE_TREND),
    'B_rms_swap': ('median', 'rms',        'q25', 'q75') + tuple(BASE_TREND),
    'C_plus_fft': ('median', 'rms',        'q25', 'q75') + tuple(BASE_TREND) + ('fft_0','fft_1','fft_2','fft_3','fft_4'),
    'D_plus_ent': ('median', 'rms',        'q25', 'q75') + tuple(BASE_TREND) + ('perm_entropy',),
}

# ---------------------------------------------------------------------------
# Redundancy check — correlation between rms and abs_energy
# ---------------------------------------------------------------------------
print("\n" + "─" * 60)
print("REDUNDANCY CHECK — rms vs abs_energy")
print("─" * 60)

abs_energy = np.sum(X_3d ** 2, axis=1).flatten()
rms_vals   = np.sqrt(np.mean(X_3d ** 2, axis=1)).flatten()
correlation = np.corrcoef(abs_energy, rms_vals)[0, 1]
print(f"  Pearson correlation(abs_energy, rms): {correlation:.6f}")
print(f"  abs_energy range: {abs_energy.min():.2f} → {abs_energy.max():.2f}")
print(f"  rms range:        {rms_vals.min():.4f} → {rms_vals.max():.4f}")
print(f"  → {'REDUNDANT (r>0.99)' if abs(correlation) > 0.99 else 'NOT redundant'}")

# ---------------------------------------------------------------------------
# Timing and scale per feature set
# ---------------------------------------------------------------------------
print("\n" + "─" * 60)
print("TIMING AND SCALE PER FEATURE SET")
print("─" * 60)

results = {}

for name, features in FEATURE_SETS.items():
    t0 = time.perf_counter()

    # Build feature matrix for this set
    # Split into standard features and custom ones
    std_features = [f for f in features if f in ALL_FEATURES]
    custom = [f for f in features if f not in ALL_FEATURES]

    blocks = []

    # Standard features via _compute_features
    if std_features:
        X_std = _compute_features(X_3d, std_features)
        blocks.append(X_std)

    # Custom features
    for feat in custom:
        if feat == 'rms':
            blocks.append(compute_rms(X_3d))
        elif feat.startswith('fft_'):
            k = int(feat.split('_')[1])
            fft_vals = np.fft.rfft(X_3d, axis=1)
            blocks.append(np.abs(fft_vals[:, k, :]))
        elif feat == 'perm_entropy':
            print(f"    [{name}] Computing permutation entropy (slow)...",
                  end=' ', flush=True)
            blocks.append(compute_permutation_entropy(X_3d))

    X_feat = np.concatenate(blocks, axis=1)
    elapsed = time.perf_counter() - t0

    results[name] = {'X': X_feat, 'time': elapsed, 'n_features': X_feat.shape[1]}

    print(f"\n  [{name}]")
    print(f"    Features:   {len(features)} types × {n_sensors} sensors = {X_feat.shape[1]} cols")
    print(f"    Time:       {elapsed:.2f}s")
    print(f"    Range:      {X_feat.min():.4f} → {X_feat.max():.4f}")
    print(f"    Std range:  {X_feat.std(axis=0).min():.4f} → {X_feat.std(axis=0).max():.4f}")

# ---------------------------------------------------------------------------
# PCA variance comparison
# ---------------------------------------------------------------------------
print("\n" + "─" * 60)
print("PCA EXPLAINED VARIANCE COMPARISON")
print("─" * 60)

for n_comp in [10, 15]:
    print(f"\n  n_components = {n_comp}:")
    for name, res in results.items():
        X = res['X']
        # Simulate DimReducer (RobustScaler + PCA)
        from sklearn.preprocessing import RobustScaler
        from sklearn.decomposition import PCA
        X_scaled = RobustScaler().fit_transform(X)
        pca = PCA(n_components=n_comp)
        pca.fit(X_scaled)
        cumvar = pca.explained_variance_ratio_.cumsum()[-1]
        pc1 = pca.explained_variance_ratio_[0]
        print(f"    {name:<16}: cumvar={cumvar:.3f}, PC1={pc1:.3f}")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  {'Set':<16} {'Time(s)':>8} {'n_feat':>7} {'Speedup vs A':>14}")
t_a = results['A_current']['time']
for name, res in results.items():
    speedup = t_a / res['time'] if res['time'] > 0 else 0
    marker = '← baseline' if name == 'A_current' else f'×{speedup:.1f} faster' if speedup > 1 else f'×{1/speedup:.1f} slower'
    print(f"  {name:<16} {res['time']:>8.2f} {res['n_features']:>7} {marker:>14}")

FEATURE EXPLORATION — Motor 1

Windows: 163, window_size: 30, sensors: 16

────────────────────────────────────────────────────────────
REDUNDANCY CHECK — rms vs abs_energy
────────────────────────────────────────────────────────────
  Pearson correlation(abs_energy, rms): 0.974466
  abs_energy range: 0.00 → 2457972797.70
  rms range:        0.0002 → 9051.6533
  → NOT redundant

────────────────────────────────────────────────────────────
TIMING AND SCALE PER FEATURE SET
────────────────────────────────────────────────────────────

  [A_current]
    Features:   12 types × 16 sensors = 192 cols
    Time:       11.79s
    Range:      -0.7977 → 2457972797.7024
    Std range:  0.0000 → 1617552.5494

  [B_rms_swap]
    Features:   12 types × 16 sensors = 192 cols
    Time:       9.00s
    Range:      -0.7977 → 9055.3975
    Std range:  0.0000 → 6.8673

  [C_plus_fft]
    Features:   17 types × 16 sensors = 272 cols
    Time:       5.00s
    Range:      -0.7977 → 271549.5600
    Std range:  

In [2]:
"""Validation script for GGS I/O — checkpointing and resume logic.

This script validates the complete GGS I/O pipeline in four phases:

    Phase 1 — Fresh run (partial):
        Runs GGS with checkpoint_every=1 but interrupts after N configs
        by monkey-patching _run_single_config to raise after a threshold.
        Verifies: metadata.json created, checkpoint.csv exists with partial results.

    Phase 2 — Resume:
        Calls group_grid_search() again with the same param_grid.
        Verifies: checkpoint detected, timestamp inherited, prior results loaded,
        pending configs are only the ones not yet evaluated.

    Phase 3 — Completion:
        The resumed run completes all remaining configs.
        Verifies: results.csv has ALL configs (prior + new), checkpoint removed.

    Phase 4 — New param_grid:
        Calls group_grid_search() with a different param_grid.
        Verifies: new session created (different hash), previous results untouched.

Run from project root:
    python pipeline_validation_ggs_io.py
"""

import json
import shutil
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Setup — import project modules
# ---------------------------------------------------------------------------
try:
    # Si __file__ existe (script normal)
    project_root = Path(__file__).parent.parent
except NameError:
    # Entorno interactivo: asume que el directorio actual es el raíz del proyecto
    # O usa una ruta fija conocida
    project_root = Path.cwd().parent  # o Path('.').absolute()

from src.dataset_manager import DatasetManager
from src.ggs_training_manager import GGSTrainingManager
from src.models.svr_model import SVRModel
from src.utils.ggs_io import (
    compute_param_grid_hash,
    resolve_ggs_session,
)
import src.ggs_training_manager as ggs_module

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

BASE_DIR = Path('outputs/ggs_validation_test')  # isolated from real outputs
N_FOLDS  = 2

PARAM_GRID = {
    'feature_set':        ['A', 'B'],
    'window_size':        [20],
    'n_components':       [5],
    'clipping_threshold': [125],
    'C':                  [1.0],
    'kernel':             ['rbf'],
    'epsilon':            [0.1],
}
# Total configs = 2 feature_sets × 1 × 1 × 1 × 1 × 1 × 1 = 2

PARAM_GRID_NEW = {
    **PARAM_GRID,
    'window_size': [25],  # different → different hash → new session
}

TOTAL_CONFIGS = 2  # product of all param_grid values

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _sep(title: str) -> None:
    print(f"\n{'─'*60}")
    print(f"  {title}")
    print(f"{'─'*60}")


def _ok(msg: str) -> None:
    print(f"  ✅  {msg}")


def _fail(msg: str) -> None:
    print(f"  ❌  {msg}")
    sys.exit(1)


def _check(condition: bool, ok_msg: str, fail_msg: str) -> None:
    if condition:
        _ok(ok_msg)
    else:
        _fail(fail_msg)


# ---------------------------------------------------------------------------
# Load data — motors 1, 2, 3 (minimum for GroupKFold n_splits=2)
# ---------------------------------------------------------------------------

_sep("Loading data (motors 1, 2, 3)")

dfs = []
for motor_id in [1, 2, 3]:
    import pandas as _pd
    df = _pd.read_csv(f'data/clean/data_motor_{motor_id}.csv')
    df.insert(0, 'unit_number', motor_id)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
X_df   = df_all.drop(columns=['RUL'])
y_df   = df_all[['unit_number', 'time_in_cycles', 'RUL']].copy()
groups = df_all['unit_number'].to_numpy()

print(f"  Motors: {X_df['unit_number'].nunique()}, Rows: {len(X_df)}")

# ---------------------------------------------------------------------------
# Clean up any leftover test artifacts
# ---------------------------------------------------------------------------

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
    print(f"  Cleaned up previous test dir: {BASE_DIR}")

# ---------------------------------------------------------------------------
# Phase 1 — Partial run (simulate interruption after first config)
# ---------------------------------------------------------------------------

_sep("Phase 1 — Partial run (interrupt after 1 config)")

# Monkey-patch _run_single_config to raise after N_INTERRUPT configs
_call_count = 0
_N_INTERRUPT = 1
_original_run = ggs_module._run_single_config

def _patched_run(*args, **kwargs):
    global _call_count
    _call_count += 1
    if _call_count > _N_INTERRUPT:
        raise KeyboardInterrupt("Simulated interruption for testing")
    return _original_run(*args, **kwargs)

ggs_module._run_single_config = _patched_run

manager = GGSTrainingManager(
    model_class=SVRModel,
    X_df=X_df,
    y_df=y_df,
    groups=groups,
)

# Capture the session before running to get expected paths
param_hash    = compute_param_grid_hash(PARAM_GRID)
expected_stem = f'SVRModel_{param_hash}'

try:
    manager.group_grid_search(
        param_grid=PARAM_GRID,
        n_folds=N_FOLDS,
        silence=True,
        checkpoint_every=1,
        base_dir=BASE_DIR,
    )
except KeyboardInterrupt:
    print("  ⚡ Interruption detected (expected)")

# Restore original function
ggs_module._run_single_config = _original_run
_call_count = 0

# Verify Phase 1
metadata_files   = list((BASE_DIR / 'metadata').glob(f'{expected_stem}_*.json'))
checkpoint_files = list((BASE_DIR / 'checkpoints').glob(f'{expected_stem}_*.csv'))
results_files    = list((BASE_DIR / 'results').glob(f'{expected_stem}_*.csv'))

_check(len(metadata_files) == 1,
       f"metadata.json created: {metadata_files[0].name if metadata_files else '—'}",
       f"metadata.json NOT found in {BASE_DIR / 'metadata'}")

_check(len(checkpoint_files) == 1,
       f"checkpoint.csv created: {checkpoint_files[0].name if checkpoint_files else '—'}",
       f"checkpoint.csv NOT found in {BASE_DIR / 'checkpoints'}")

_check(len(results_files) == 0,
       "results.csv NOT yet created (correct — run incomplete)",
       "results.csv created prematurely")

if checkpoint_files:
    ckpt_df = pd.read_csv(checkpoint_files[0])
    _check(len(ckpt_df) == _N_INTERRUPT,
           f"checkpoint has {_N_INTERRUPT} row(s) (correct)",
           f"checkpoint has {len(ckpt_df)} rows — expected {_N_INTERRUPT}")

# Store timestamp for Phase 2 comparison
original_timestamp = checkpoint_files[0].stem.split(param_hash + '_')[1] if checkpoint_files else None
print(f"  Session timestamp: {original_timestamp}")

# ---------------------------------------------------------------------------
# Phase 2 — Resume detection
# ---------------------------------------------------------------------------

_sep("Phase 2 — Resume detection")

session_resumed = resolve_ggs_session(
    model_class=SVRModel,
    param_grid=PARAM_GRID,
    base_dir=BASE_DIR,
)

_check(session_resumed.is_resume is True,
       "Session detected as resume",
       "Session NOT detected as resume")

_check(session_resumed.timestamp == original_timestamp,
       f"Timestamp inherited: {session_resumed.timestamp}",
       f"Timestamp NOT inherited — got {session_resumed.timestamp}, expected {original_timestamp}")

_check(len(session_resumed.prior_results) == _N_INTERRUPT,
       f"Prior results loaded: {len(session_resumed.prior_results)} row(s)",
       f"Prior results count mismatch: {len(session_resumed.prior_results)}")

_check(len(session_resumed.completed_keys) == _N_INTERRUPT,
       f"Completed keys populated: {len(session_resumed.completed_keys)} key(s)",
       f"Completed keys mismatch: {len(session_resumed.completed_keys)}")

pending = TOTAL_CONFIGS - len(session_resumed.completed_keys)
_check(pending == TOTAL_CONFIGS - _N_INTERRUPT,
       f"Pending configs: {pending} (correct)",
       f"Pending configs: {pending} — expected {TOTAL_CONFIGS - _N_INTERRUPT}")

# ---------------------------------------------------------------------------
# Phase 3 — Complete the run (resume)
# ---------------------------------------------------------------------------

_sep("Phase 3 — Complete run (resume)")

manager2 = GGSTrainingManager(
    model_class=SVRModel,
    X_df=X_df,
    y_df=y_df,
    groups=groups,
)

all_results = manager2.group_grid_search(
    param_grid=PARAM_GRID,
    n_folds=N_FOLDS,
    silence=True,
    checkpoint_every=1,
    base_dir=BASE_DIR,
)

# Verify Phase 3
results_files_after = list((BASE_DIR / 'results').glob(f'{expected_stem}_*.csv'))
checkpoint_files_after = list((BASE_DIR / 'checkpoints').glob(f'{expected_stem}_*.csv'))

_check(len(results_files_after) == 1,
       f"results.csv created: {results_files_after[0].name if results_files_after else '—'}",
       "results.csv NOT created after completion")

_check(len(checkpoint_files_after) == 0,
       "checkpoint.csv removed after completion",
       f"checkpoint.csv still exists: {checkpoint_files_after}")

if results_files_after:
    results_df = pd.read_csv(results_files_after[0])
    _check(len(results_df) == TOTAL_CONFIGS,
           f"results.csv has {TOTAL_CONFIGS} rows (all configs present)",
           f"results.csv has {len(results_df)} rows — expected {TOTAL_CONFIGS}")

    _check('feature_set' in results_df.columns,
           "feature_set column present in results",
           "feature_set column MISSING from results")

    _check(set(results_df['feature_set'].unique()) == {'A', 'B'},
           "Both feature sets (A, B) present in results",
           f"feature_sets in results: {set(results_df['feature_set'].unique())}")

_check(len(all_results) == TOTAL_CONFIGS,
       f"manager.ggs_results_ has {TOTAL_CONFIGS} entries",
       f"manager.ggs_results_ has {len(all_results)} entries")

# Verify timestamp consistency
if results_files_after:
    result_timestamp = results_files_after[0].stem.split(param_hash + '_')[1]
    _check(result_timestamp == original_timestamp,
           f"results.csv timestamp matches original: {result_timestamp}",
           f"Timestamp mismatch: {result_timestamp} != {original_timestamp}")

# ---------------------------------------------------------------------------
# Phase 4 — New param_grid → new session
# ---------------------------------------------------------------------------

_sep("Phase 4 — New param_grid → new session")

manager3 = GGSTrainingManager(
    model_class=SVRModel,
    X_df=X_df,
    y_df=y_df,
    groups=groups,
)

new_results = manager3.group_grid_search(
    param_grid=PARAM_GRID_NEW,
    n_folds=N_FOLDS,
    silence=True,
    checkpoint_every=1,
    base_dir=BASE_DIR,
)

new_hash = compute_param_grid_hash(PARAM_GRID_NEW)
new_results_files = list((BASE_DIR / 'results').glob(f'SVRModel_{new_hash}_*.csv'))
old_results_files = list((BASE_DIR / 'results').glob(f'{expected_stem}_*.csv'))

_check(new_hash != param_hash,
       f"New param_grid produces different hash: {new_hash} ≠ {param_hash}",
       "New param_grid produced SAME hash — collision!")

_check(len(new_results_files) == 1,
       f"New session results.csv created: {new_results_files[0].name if new_results_files else '—'}",
       "New session results.csv NOT created")

_check(len(old_results_files) == 1,
       "Original results.csv untouched",
       "Original results.csv MISSING after new session")

_check(len(new_results) == 2,
       f"New session has 2 results (window_size=25 × 2 feature_sets)",
       f"New session has {len(new_results)} results")

# ---------------------------------------------------------------------------
# Phase 5 — Metadata content verification
# ---------------------------------------------------------------------------

_sep("Phase 5 — Metadata content verification")

metadata_files = list((BASE_DIR / 'metadata').glob(f'{expected_stem}_*.json'))
if metadata_files:
    with open(metadata_files[0]) as f:
        meta = json.load(f)

    _check(meta['model'] == 'SVRModel',
           f"model in metadata: {meta['model']}",
           f"model in metadata wrong: {meta['model']}")

    _check(meta['n_folds'] == N_FOLDS,
           f"n_folds in metadata: {meta['n_folds']}",
           f"n_folds in metadata wrong: {meta['n_folds']}")

    _check(meta['total_configs'] == TOTAL_CONFIGS,
           f"total_configs in metadata: {meta['total_configs']}",
           f"total_configs in metadata wrong: {meta['total_configs']}")

    _check('param_grid' in meta,
           "param_grid present in metadata",
           "param_grid MISSING from metadata")

    _check(meta['param_hash'] == param_hash,
           f"param_hash in metadata: {meta['param_hash']}",
           f"param_hash wrong: {meta['param_hash']}")
else:
    _fail("metadata.json not found for verification")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

_sep("VALIDATION SUMMARY")
print("""
  Phase 1 — Partial run + checkpoint creation  ✅
  Phase 2 — Resume detection + timestamp inherit ✅
  Phase 3 — Completion + results + checkpoint cleanup ✅
  Phase 4 — New param_grid → new session ✅
  Phase 5 — Metadata content ✅

  GGS I/O pipeline is ready for production use.
""")

# Clean up test artifacts
shutil.rmtree(BASE_DIR)
print(f"  Test artifacts cleaned up: {BASE_DIR}")


────────────────────────────────────────────────────────────
  Loading data (motors 1, 2, 3)
────────────────────────────────────────────────────────────
  Motors: 3, Rows: 658

────────────────────────────────────────────────────────────
  Phase 1 — Partial run (interrupt after 1 config)
────────────────────────────────────────────────────────────
  🆕 New GGS session: SVRModel_1214f7aa_20260507_2030
GGS: 2 total configs × 2 folds


GGS SVRModel:  50%|█████     | 1/2 [00:00<00:00,  3.25it/s]


  ⚡ Interruption detected (expected)
  ✅  metadata.json created: SVRModel_1214f7aa_20260507_2030.json
  ✅  checkpoint.csv created: SVRModel_1214f7aa_20260507_2030.csv
  ✅  results.csv NOT yet created (correct — run incomplete)
  ✅  checkpoint has 1 row(s) (correct)
  Session timestamp: 20260507_2030

────────────────────────────────────────────────────────────
  Phase 2 — Resume detection
────────────────────────────────────────────────────────────
  ♻️  Resuming session: SVRModel_1214f7aa_20260507_2030.csv
     Configs completed: 1
  ✅  Session detected as resume
  ✅  Timestamp inherited: 20260507_2030
  ✅  Prior results loaded: 1 row(s)
  ✅  Completed keys populated: 1 key(s)
  ✅  Pending configs: 1 (correct)

────────────────────────────────────────────────────────────
  Phase 3 — Complete run (resume)
────────────────────────────────────────────────────────────
  ♻️  Resuming session: SVRModel_1214f7aa_20260507_2030.csv
     Configs completed: 1
GGS: 2 total configs × 2 folds
     

GGS SVRModel: 100%|██████████| 1/1 [00:00<00:00,  3.15it/s]


Checkpoint removed: SVRModel_1214f7aa_20260507_2030.csv
Results saved: outputs/ggs_validation_test/results/SVRModel_1214f7aa_20260507_2030.csv
  ✅  results.csv created: SVRModel_1214f7aa_20260507_2030.csv
  ✅  checkpoint.csv removed after completion
  ✅  results.csv has 2 rows (all configs present)
  ✅  feature_set column present in results
  ✅  Both feature sets (A, B) present in results
  ✅  manager.ggs_results_ has 2 entries
  ✅  results.csv timestamp matches original: 20260507_2030

────────────────────────────────────────────────────────────
  Phase 4 — New param_grid → new session
────────────────────────────────────────────────────────────
  🆕 New GGS session: SVRModel_62bf4a44_20260507_2030
GGS: 2 total configs × 2 folds


GGS SVRModel: 100%|██████████| 2/2 [00:00<00:00,  3.59it/s]

Checkpoint removed: SVRModel_62bf4a44_20260507_2030.csv
Results saved: outputs/ggs_validation_test/results/SVRModel_62bf4a44_20260507_2030.csv
  ✅  New param_grid produces different hash: 62bf4a44 ≠ 1214f7aa
  ✅  New session results.csv created: SVRModel_62bf4a44_20260507_2030.csv
  ✅  Original results.csv untouched
  ✅  New session has 2 results (window_size=25 × 2 feature_sets)

────────────────────────────────────────────────────────────
  Phase 5 — Metadata content verification
────────────────────────────────────────────────────────────
  ✅  model in metadata: SVRModel
  ✅  n_folds in metadata: 2
  ✅  total_configs in metadata: 2
  ✅  param_grid present in metadata
  ✅  param_hash in metadata: 1214f7aa

────────────────────────────────────────────────────────────
  VALIDATION SUMMARY
────────────────────────────────────────────────────────────

  Phase 1 — Partial run + checkpoint creation  ✅
  Phase 2 — Resume detection + timestamp inherit ✅
  Phase 3 — Completion + results + che